In [1]:
import pandas as pd 
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration


In [2]:
train_data=pd.read_csv("dataset/samsum-train.csv")
val_data=pd.read_csv("dataset/samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.sample(6)

,id,dialogue,summary
10093,13862796,Giles: at what time is PE class?\nHenry: 10:45...,Giles' PE class is at 10:45.
2712,13829590,Hannah: Heey\r\nHannah: How's your weekend goi...,Nadia wasn't feeling well on Friday. Hannah is...
13092,13594099-1,"Thierry: Hello young people, how are you? We a...",Thierry and Cecile want to travel to Rome a we...
11559,13681827,Mandy: Can't go with u after class\r\nKaren: W...,"Mandy can't go with Karen after class, as she ..."
987,13727964,Feyi: When will you be able to work on the Dec...,Kurt will finish December's email and blog pos...
9029,13728565,"Don: I have to mention, I really didn't like y...",Don and Sally don't like Sally's sister's boyf...


In [5]:
train_data.shape

(14732, 3)

In [6]:
val_data.shape

(818, 3)

In [7]:
# random sampling 
train_data=train_data.sample(4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(500,random_state=42).reset_index(drop=True)

In [8]:
train_data.shape
val_data.shape

(500, 3)

### Data Preprocessing 

In [9]:
import re

In [10]:
def clean_data(text):
    text=re.sub(r"\r\n"," ",text) # lines
    text=re.sub(r"\s+", " ",text) # Spaces
    text=re.sub(r"<.*?>"," ",text) # html tags 
    text.strip()
    text.lower()
    return text

In [11]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)



In [12]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

### Tokenize

In [13]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [14]:
# raw data => tokenized inputs for fine tuning 

def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"]=targets["input_ids"]
    return inputs

In [15]:
train_dataset=train_data.apply(tokenize,axis=1).to_list()

In [16]:
val_dataset=val_data.apply(tokenize,axis=1).to_list()

In [17]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [18]:
# input ids

# 1 => EOS, 0 => padding

# attention masks => Indicates which are the valid ids 

# labels-targets => summary token

In [19]:
len(train_dataset[0]["input_ids"])

512

In [20]:
len(train_dataset[0]["labels"])

150

In [21]:
type(train_dataset)

list

In [22]:
type(val_dataset)

list

### Working with our Model

In [23]:
# NLP => Generation task 

model=T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Fine-Tune

In [24]:
import torch
from transformers import is_av_available

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

In [25]:
print(device)

cuda


In [26]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [27]:
# Training Arguments

from numpy import save


training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr in 500

)

In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [29]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.074144,0.392600
2,0.408071,0.367184
3,0.384726,0.359581
4,0.372528,0.356459
5,0.364883,0.355554
6,0.361752,0.355093


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9943505350748698, metrics={'train_runtime': 935.6687, 'train_samples_per_second': 25.65, 'train_steps_per_second': 3.206, 'total_flos': 3248203235328000.0, 'train_loss': 0.9943505350748698, 'epoch': 6.0})

In [30]:
# load model => fine tune => save tune the model

In [31]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [32]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Testing Core logic for summarization

In [33]:


def summarize_dialogue(dialogue):

    dialogue=clean_data(dialogue) # clean

    # tokenize
    inputs=tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )


    #generate the summary => token_ids
    targets=model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # convert token_ids to text => decoding 
    summary=tokenizer.decode(targets[0],skip_special_tokens=True) #EOS, TAB, SEP
    return summary

In [35]:
print(type(tokenizer))
print(tokenizer.vocab_file)

<class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>


AttributeError: T5Tokenizer has no attribute vocab_file

In [36]:
import transformers
print(transformers.__version__)

5.14.1


In [34]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  AI adoption has significantly increased over the past few years. Expert is concerned about bias in AI models because they reflect the data they are trained on.
